<h1 style="background: linear-gradient(to right, #395761, #7799B6);
           color: white;
           padding: 22px;
           border-left: 10px solid #CEEB13;
           text-align: center;
           font-family: 'Roboto Mono', Menlo, 'Courier New', monospace;
           font-weight: 600;">
  1. Deploy SGLang on your GPU
</h1>

You deploy [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) as a production-style endpoint on your own NVIDIA H200: one Kubernetes object, served by [SGLang](https://docs.sglang.io/) through [KServe](https://kserve.github.io/website/) on [Crusoe Managed Kubernetes](https://docs.crusoe.ai/orchestration/cmk/overview). Then you talk to it, look inside the engine, take a baseline benchmark and put it on the results wall. Part 2 ([`02_Tune.ipynb`](02_Tune.ipynb)) tunes it.

| Step | What you do | Time |
|---|---|---|
| Start (before the talk) | Run the next three cells: check access, deploy, status | 2 min |
| While it comes up (during the talk) | Read how the manifest works | — |
| 1. Ready | Wait until your server is up | 0–5 min |
| 2. Talk to the model | Chat with thinking off and on | 3 min |
| 3. Look inside | Server arguments, KV cache size, metrics | 2 min |
| 4. Baseline benchmark | The standard benchmark (256 users), then a single user for comparison | 3 min |
| 5. Results wall | Submit your baseline | 1 min |
| 6. Chat in a browser | A one-page chat UI | 2 min |

## Start here: run the next three cells now

The kubeconfig `login.sh` saved points `kubectl` at your own namespace (`ws-NN`) with one GPU. The first cell checks that, the second asks KServe to create your model server, the third shows where it is. Then leave it alone: it takes a few minutes, more when everyone deploys at the same moment, and it comes up during the talk. The model download sometimes takes up to 10 minutes; read the next section while it loads.

In [ ]:
import sys

sys.path.insert(0, "..")

from workshop_utils import (
    PortForward,
    chat,
    deploy,
    k8s,
    list_models,
    metrics,
    participant_config,
    participant_namespace,
    participant_profile,
    render,
    server_info,
    submit,
    to_yaml,
    wait_ready,
)
from workshop_utils.bench import STANDARD, run_bench, save_results, standard_bench, table
from workshop_utils.plots import compare_bars

NAMESPACE = participant_namespace()  # stops with a clear message if your kubeconfig has no namespace


def can_i(*what: str) -> str:
    try:
        return k8s.kubectl("auth", "can-i", *what, "-n", NAMESPACE).strip()
    except RuntimeError:  # kubectl exits 1 on "no"
        return "no"


print("namespace:", NAMESPACE)
for what in (("create", "llminferenceservices.serving.kserve.io"), ("create", "pods/exec"), ("create", "pods/portforward")):
    print(f"can-i {' '.join(what):<48} {can_i(*what)}")
gpu = k8s.kubectl("get", "resourcequota", "gpu-quota", "-n", NAMESPACE,
                  "-o", r"jsonpath={.status.used.requests\.nvidia\.com/gpu}/{.status.hard.requests\.nvidia\.com/gpu}")
print(f"GPU in use: {gpu}")

In [ ]:
prof = participant_profile()  # the H200 pool of the workshop cluster
cfg = participant_config(NAMESPACE)  # service "qwen3": Qwen3-8B, 1 GPU, SGLang defaults otherwise

if k8s.exists_llmisvc(cfg.name, NAMESPACE):
    # Re-applying a changed config would never finish (one GPU): part 2 uses redeploy() for that.
    print(f"{cfg.name} already exists in {NAMESPACE}; not deploying again.")
else:
    deploy(prof, cfg)

In [ ]:
print(k8s.status_line(cfg.name, NAMESPACE))  # re-run this cell any time to see where your server is

## While it comes up

Nothing to run until [step 1](#1.-Wait-until-your-server-is-Ready). Read on: this is what you just deployed.

### Three things to know

**SGLang** is an open-source inference engine for large language models. It takes a model's weights, keeps the KV cache of every in-flight request in GPU memory, batches requests together on every decode step, and serves the result behind an OpenAI-compatible HTTP API. Its prefix cache, [RadixAttention](https://lmsys.org/blog/2024-01-17-sglang/), reuses the KV cache of any shared prompt prefix, which is what makes chat and agent workloads fast. The same engine code runs on NVIDIA (CUDA) and AMD (ROCm); only the kernel layer underneath differs. Docs: [docs.sglang.io](https://docs.sglang.io/), source: [sgl-project/sglang](https://github.com/sgl-project/sglang).

**KServe** is a Kubernetes layer for model serving. Its [LLMInferenceService](https://kserve.github.io/website/docs/model-serving/generative-inference/llmisvc/llmisvc-overview) resource describes an LLM endpoint in one object: where the weights are, which container serves them, how many GPUs. The controller turns it into a Deployment, a pod and a Service, and downloads the weights before the server starts. It handles one service at a time, so when everyone deploys together your pod waits its turn.

**Crusoe Managed Kubernetes** (CMK) is a managed Kubernetes control plane with GPU node pools. The [NVIDIA GPU Operator](https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/) add-on lets pods ask for `nvidia.com/gpu`, and each node carries a `crusoe.ai/accelerator` label naming its GPU type. This cluster has 8 nodes with 8 H200s each; you have one of those 64 GPUs. Docs: [CMK overview](https://docs.crusoe.ai/orchestration/cmk/overview).

<div align="center"><img src="../assets/architecture.svg" alt="Your laptop applies a manifest to a Crusoe Managed Kubernetes cluster. KServe creates a pod that downloads the weights and runs an SGLang server on one GPU. Only the image, the GPU resource name and the node label differ between NVIDIA and AMD." width="900"/></div>

### The manifest you deployed

A `Profile` holds the vendor-specific values and a `ServingConfig` holds everything about how the model is served. Rendering the two gives the `LLMInferenceService` you applied; [`../manifests/sglang-qwen3-nvidia.yaml`](../manifests/sglang-qwen3-nvidia.yaml) is the same object for a `kubectl apply` without this notebook.

In [ ]:
print(to_yaml(render(prof, cfg)))

Read it top to bottom:

| Block | What it does |
|---|---|
| `metadata.name: qwen3`, `namespace` | Your service, in your namespace. |
| `spec.model.uri: hf://Qwen/Qwen3-8B` | KServe's storage-initializer downloads the 16 GB of weights into `/mnt/models` before the server starts (about a minute). |
| `spec.model.name: qwen3` | The name clients use in `model=` and the one `/v1/models` reports. |
| `containers[main].image` | `lmsysorg/sglang:v0.5.22-runtime`: the official SGLang build, pinned to one release. |
| `command` and `args` | `python3 -m sglang.launch_server` with explicit flags: `--tp-size 1` is one GPU; `--enable-metrics` exposes `/metrics`; `--reasoning-parser qwen3` separates Qwen3's thinking from its answer. Part 2 changes these. Reference: [server arguments](https://docs.sglang.io/advanced_features/server_arguments.html). |
| `env: HF_HUB_OFFLINE=1` | The weights are already on disk; the server never contacts the Hub. |
| `resources` | One `nvidia.com/gpu`, 8 CPUs and 64Gi of memory, requests equal to limits. |
| `startupProbe` on `/health` | Loading and CUDA graph capture take a minute or two; the probe allows up to 30 minutes before Kubernetes gives up. |
| `volumes[dshm]` | Shared memory for tensor-parallel ranks; it matters once a server spans several GPUs. |
| `nodeSelector` | `crusoe.ai/accelerator: nvidia-h200-141gb-sxm-ib` pins the pod to the H200 pool. |

No `router` block: you reach the server with `kubectl port-forward`, which goes through the Kubernetes API and needs no public endpoint.

### What changes on AMD

The instructors run the same manifest on an AMD MI355X. The whole difference:

```diff
-      image: lmsysorg/sglang:v0.5.22-runtime
+      image: lmsysorg/sglang:v0.5.22-rocm724-mi35x
+      - name: SGLANG_USE_AITER
+        value: '1'
-          nvidia.com/gpu: '1'
+          amd.com/gpu: '1'
-      crusoe.ai/accelerator: nvidia-h200-141gb-sxm-ib
+      crusoe.ai/accelerator: amd-mi355x-288gb-roce
```

SGLang publishes a ROCm build of every release ([AMD guide](https://docs.sglang.io/platforms/amd_gpu.html)); the pod asks for `amd.com/gpu`, which the AMD GPU Operator advertises; `SGLANG_USE_AITER=1` turns on AMD's [AITER](https://github.com/ROCm/aiter) kernels. The flags, the API and the benchmark are the same.

## 1. Wait until your server is Ready

This cell polls every 15 seconds. Expect, in order: "waiting for KServe to create your pod" (your turn in KServe's queue), `Init:storage-initializer` (the download, about 1 minute), `Running` (loading the weights and capturing CUDA graphs, about 1.5 minutes), then `Ready`. If it was already Ready during the talk, this returns at once. The model download sometimes takes up to 10 minutes instead of one; read the next section while it loads.

In [ ]:
ready = wait_ready(cfg.name, NAMESPACE, timeout_s=1800)

The server log tells the story: the weights load, the KV cache pool is sized from the GPU memory left over, graphs are captured, and finally `The server is fired up and ready to roll!`. If the pod never gets Ready, the last log lines are the first place to look.

In [ ]:
print(k8s.logs(cfg.name, NAMESPACE, tail=15))

## 2. Talk to the model

`PortForward` runs `kubectl port-forward` to your server's Service and gives you a local URL. The endpoint speaks the OpenAI API, so `list_models` is a plain `GET /v1/models` and `chat` uses the [openai](https://github.com/openai/openai-python) Python client. Qwen3 has a thinking mode: the first call turns it off through the chat template, the second turns it on and shows the reasoning that the `qwen3` parser splits out. Reference: [OpenAI-compatible API](https://docs.sglang.io/basic_usage/openai_api_completions.html).

In [ ]:
with PortForward(cfg.name, NAMESPACE) as url:
    print("models:", list_models(url))
    r = chat(url, cfg.served_model_name, "In two sentences, what is a KV cache and why does an LLM server keep one?")

In [ ]:
with PortForward(cfg.name, NAMESPACE) as url:
    r = chat(url, cfg.served_model_name, "Is 221 a prime number? Answer yes or no, then one line of justification.",
             thinking=True, max_tokens=800)
    print("\n--- reasoning (first 600 characters) ---\n", r.reasoning[:600])

The same call from a terminal (after `export KUBECONFIG=~/.sglang-workshop/kubeconfig`, which `login.sh` saved):

```bash
kubectl port-forward svc/qwen3-kserve-workload-svc 8001:8000 &
curl -s http://127.0.0.1:8001/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model": "qwen3", "messages": [{"role": "user", "content": "Hello"}], "max_tokens": 64}'
```

## 3. Look inside the engine

Two native endpoints are worth knowing. `/get_server_info` returns the server's arguments and the size of its KV pool, which is how you confirm a flag took effect. `/metrics` is the Prometheus endpoint; `sglang:cache_hit_rate` and `sglang:token_usage` are the numbers to watch when you tune. Reference: [native API](https://docs.sglang.io/basic_usage/native_api.html), [production metrics](https://docs.sglang.io/references/production_metrics.html).

In [ ]:
with PortForward(cfg.name, NAMESPACE) as url:
    info = server_info(url)
    m = metrics(url)

keys = ["tp_size", "mem_fraction_static", "kv_cache_dtype", "schedule_policy", "chunked_prefill_size",
        "max_running_requests", "enable_hierarchical_cache", "max_total_num_tokens"]
for k in keys:
    print(f"{k:<28} {info.get(k)}")
print()
for k in ("sglang:cache_hit_rate", "sglang:token_usage", "sglang:num_running_reqs", "sglang:prompt_tokens_total",
          "sglang:generation_tokens_total"):
    print(f"{k:<32} {m.get(k)}")

`max_total_num_tokens` is the KV cache pool, in tokens: how many tokens of context the server can hold across all in-flight requests at once. It is set by the GPU memory left after the weights (about 803,000 tokens here). Remember it: the benchmark below needs more than that, and the first experiment in part 2 doubles it.

## 4. Baseline benchmark

SGLang ships a load generator, `sglang.bench_serving`, in the same image. It runs *inside* your pod against `127.0.0.1:8000`, so your laptop's connection is not part of the measurement. Each run is a closed loop: a fixed number of requests are always in flight, like users who send the next prompt the moment the previous answer finishes. Every prompt is exactly 4096 random token ids and every answer exactly 512 tokens. The prompts are generated in the pod (`random-ids`); the pod cannot download datasets. Reference: [benchmark and profiling](https://docs.sglang.io/developer_guide/benchmark_and_profiling.html).

The cell below runs the **standard benchmark**, the one that goes on the results wall and that every experiment in part 2 repeats, the same for every participant: 384 requests, 256 at a time, after a short warm-up (about 2 minutes). Then it runs 8 requests one at a time, for the speed a single user sees (about 30 seconds).

<div align="center"><img src="../assets/serving-metrics.svg" alt="One request on a timeline: TTFT from request to first token, ITL between tokens, E2E latency to the last token; TPOT is decode time per token and throughput is output tokens per second across all requests." width="900"/></div>

In [ ]:
std = standard_bench(cfg.name, NAMESPACE, cfg.served_model_name, "baseline")  # the number on the results wall
one = run_bench(cfg.name, NAMESPACE, cfg.served_model_name, "1 user", **{**STANDARD, "num_prompts": 8, "max_concurrency": 1})
print(table([one, std], keys=("output_throughput", "median_ttft_ms", "p99_ttft_ms", "median_tpot_ms")))
save_results([one, std], "../results/baseline.json");

In [ ]:
compare_bars({"1 user": one, "256 users": std}, metrics=("output_throughput", "median_ttft_ms", "median_tpot_ms"),
             title=f"{prof.label}, Qwen3-8B, 4096 in / 512 out");

Read the three panels left to right:

- **Output throughput** is more than ten times higher with 256 users: every decode step reads the model's weights once and serves every sequence in the batch, so a bigger batch is almost free per token.
- **Time to first token (TTFT)** goes from a fraction of a second to many seconds. 256 requests × 4,608 tokens is about 1.18 million tokens of KV cache, more than the 803,000-token pool you saw in step 3, so not everyone fits at once: the requests that do not fit wait in the queue until others finish.
- **Time per output token (TPOT)** grows several times: each decode step now serves well over a hundred sequences, and each sequence pays for that.

That queue is what part 2 attacks first: with an FP8 KV cache all 256 fit.

## 5. Put your baseline on the results wall

`submit` stores the standard benchmark result in your namespace (a ConfigMap named `leaderboard-result`); the instructors' screen collects everyone's. Only the standard benchmark counts there.

In [ ]:
submit(std, cfg, label="baseline");

## 6. Chat in a browser

[`chat_ui.py`](chat_ui.py) next to this notebook is a one-page chat for your server: streaming answers, conversation history and a checkbox for Qwen3's thinking mode. The cell starts it in the background and prints the address; open it in a browser tab. It also runs from a terminal in the `workshop` folder: `uv run python notebooks/chat_ui.py`.

In [ ]:
import subprocess

ui = subprocess.Popen([sys.executable, "chat_ui.py"], stdout=subprocess.PIPE, text=True)
print(ui.stdout.readline().strip())  # printed once the port-forward is up

## Summary

- One `LLMInferenceService` deployed Qwen3-8B on your H200 with SGLang; KServe downloaded the weights, started the server and created its Service.
- The endpoint is OpenAI-compatible; `/get_server_info` and `/metrics` show how the engine is configured and how busy it is.
- Your baseline (the standard benchmark at 256 users, plus one user for comparison) is saved under `results/` and on the results wall.

**Keep your server running**: part 2 ([`02_Tune.ipynb`](02_Tune.ipynb)) starts from it. Stop the chat UI when you are done with it:

In [ ]:
ui.terminate()

## Further reading

- [SGLang server arguments](https://docs.sglang.io/advanced_features/server_arguments.html) and the [hyperparameter tuning guide](https://docs.sglang.io/advanced_features/hyperparameter_tuning.html)
- [KServe LLMInferenceService overview](https://kserve.github.io/website/docs/model-serving/generative-inference/llmisvc/llmisvc-overview) and [configuration reference](https://kserve.github.io/website/docs/model-serving/generative-inference/llmisvc/llmisvc-configuration)
- [Crusoe Managed Kubernetes](https://docs.crusoe.ai/orchestration/cmk/overview)
- [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B)